Make list of all 61mers present in CHM13 genome where I mask the centromere regions. Use this list of kmers as blacklist to remove centromere-unspecific kmers

In [4]:
import subprocess
import pandas as pd

In [21]:
chm13_fa = "/g/korbel/hain/reference/t2t.fa"
cent_flanks_bed = "/g/korbel/hain/centromere/intervals/cent_flanks.final.bed"
workdir = "/scratch/hain/centromere/kmer_resources/"
threads = 16

In [18]:
### get centromere flank positions
cent_flanks = pd.read_csv(
    cent_flanks_bed, 
    sep="\t", 
    header=None, names=["chrom", "start", "end", "name"]
)
### aggregate to centromere positions
cent_location = cent_flanks.groupby("chrom", as_index=False).agg(
    start=("start", "min"),
    end=("end", "max"),
)
### save as bed
cent_location.to_csv(f"{workdir}cent_location.bed", sep="\t", header=False, index=False)

In [19]:
### mask centromeres in CHM13
centromere_bed = f"{workdir}cent_location.bed"
masked_chm13_fa = f"{workdir}masked_chm13.fa"
subprocess.run(
    [
        "bedtools", 
        "maskfasta", 
        "-fi", chm13_fa, 
        "-bed", centromere_bed, 
        "-fo", masked_chm13_fa
    ]
)

CompletedProcess(args=['bedtools', 'maskfasta', '-fi', '/g/korbel/hain/reference/t2t.fa', '-bed', '/scratch/hain/centromere/kmer_resources/cent_location.bed', '-fo', '/scratch/hain/centromere/kmer_resources/masked_chm13.fa'], returncode=0)

In [22]:
### gather all 61mers in T2T
subprocess.run(
    [
        "jellyfish", "count",
        "-m", "61",
        "-s", "3G",
        "-t", str(threads),
        "-o", f"{workdir}chm13_k61.jf",
        masked_chm13_fa
    ])

CompletedProcess(args=['jellyfish', 'count', '-m', '61', '-s', '3G', '-t', '16', '-o', '/scratch/hain/centromere/kmer_resources/chm13_k61.jf', '/scratch/hain/centromere/kmer_resources/masked_chm13.fa'], returncode=0)